# ShieldWise Consolidated Evaluation Workflow

This notebook consolidates the evaluation evidence that supports the current `ShieldWise` submission.

It is intentionally aligned to the active repository implementation:

- FastAPI backend for insurance claim intake, scoring, dashboards, and alerts
- React frontend for the public view, customer dashboard, and investigator dashboard
- NLP claim-language risk scoring
- Supporting receipt-evidence model artefacts
- Regression-tested insurance workflow in `tests/test_api_insurance.py`

This notebook focuses on the evaluation artefacts that are already retained in the repository rather than inventing new model claims.

## Submission Scope

The active submission surface is the insurance workflow in:

- `src/api/`
- `src/frontend/`
- `tests/test_api_insurance.py`

The evaluation evidence consolidated here comes from:

- `backend/saved_models/nlp_metrics.json`
- `backend/receipts_models/cv_metrics.json`
- retained notebook and model artefacts in `backend/`

The runtime evidence-analysis path in the API is intentionally lighter than the retained CV research assets. This notebook keeps that distinction explicit.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "backend":
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_DIR = PROJECT_ROOT / "backend"
NLP_METRICS_PATH = BACKEND_DIR / "saved_models" / "nlp_metrics.json"
CV_METRICS_PATH = BACKEND_DIR / "receipts_models" / "cv_metrics.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"NLP metrics path: {NLP_METRICS_PATH}")
print(f"CV metrics path: {CV_METRICS_PATH}")

In [ ]:
with open(NLP_METRICS_PATH, "r", encoding="utf-8") as file:
    nlp_metrics = json.load(file)

with open(CV_METRICS_PATH, "r", encoding="utf-8") as file:
    cv_metrics = json.load(file)

nlp_metrics, cv_metrics

## NLP Evaluation Summary

The NLP component supports claim-language risk scoring. The repository retains saved NLP metrics for the trained claim-email ham/spam pipeline.

These metrics are useful as supporting evidence for the final report, but they should be interpreted carefully. The project dataset is small and should be described as a prototype workflow rather than as proof of production-grade insurer performance.

In [ ]:
nlp_rows = []
for model_name, model_metrics in nlp_metrics["models"].items():
    nlp_rows.append({"model": model_name, **model_metrics})

nlp_df = pd.DataFrame(nlp_rows).set_index("model")
nlp_df

In [ ]:
summary_df = pd.DataFrame([
    {
        "dataset_path": nlp_metrics["dataset_path"],
        "rows": nlp_metrics["rows"],
        "train_rows": nlp_metrics["train_rows"],
        "test_rows": nlp_metrics["test_rows"],
        "selected_features": nlp_metrics["selected_features"],
    }
])
summary_df

In [ ]:
ax = nlp_df[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot(
    kind="bar",
    figsize=(10, 5),
    ylim=(0, 1.05),
    title="ShieldWise NLP Model Metrics"
)
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Receipt Evidence Model Evaluation Summary

The repository also retains supporting receipt-model metrics. These are useful as background evidence for the project write-up, but the active runtime insurance workflow currently uses lightweight service-layer checks in `src/api/services/document_risk.py` rather than serving these deep models directly.

In [ ]:
cv_rows = []
for model_name, model_metrics in cv_metrics.items():
    cv_rows.append({"model": model_name, **model_metrics})

cv_df = pd.DataFrame(cv_rows).set_index("model")
cv_df

In [ ]:
plot_columns = [column for column in ["accuracy", "precision", "recall", "f1_score", "roc_auc"] if column in cv_df.columns]
ax = cv_df[plot_columns].fillna(0).plot(
    kind="bar",
    figsize=(10, 5),
    ylim=(0, 1.05),
    title="ShieldWise Supporting Receipt Model Metrics"
)
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## How These Results Fit the Final Submission

For the final ShieldWise narrative, the evaluation evidence should be framed in three layers:

1. **Active application workflow**
   The project that is actually submitted is the insurance platform implemented in `src/api/`, `src/frontend/`, and `tests/test_api_insurance.py`.

2. **Retained model evidence**
   The NLP and receipt-model metrics provide supporting evidence that the repository includes trained modelling work relevant to claim-language scoring and evidence analysis.

3. **Runtime distinction**
   The current API document-risk flow is service-based and lightweight at runtime, so the report should avoid claiming that the deep receipt models are directly deployed in the live application unless that is explicitly true in the implementation.

## Recommended Report Wording

A concise, defensible summary for the report would be:

> The ShieldWise repository retains trained NLP and receipt-analysis artefacts as supporting model evidence. The active submission surface, however, is the insurance workflow implemented through the FastAPI backend, React frontend, and integration-tested claim-monitoring flow. Model metrics are therefore presented as supporting evaluation evidence rather than as a claim that every retained model artefact is fully deployed in the runtime application.

This wording keeps the submission aligned with the repository truth.

## Limitations to State Clearly

- The retained metrics should be treated as supporting artefacts, not as a complete production evaluation package.
- The repository contains supporting CV model outputs, but the active runtime evidence scoring path is lighter and rule-based.
- The NLP metrics appear strong on the retained dataset, so the report should avoid overstating real-world generalisation without additional external validation.
- The final evaluation section should cite both the automated workflow tests and the retained model metrics, because the submission combines application engineering and supporting model work.

In [ ]:
evaluation_inventory = pd.DataFrame([
    {
        "artifact": "NLP metrics",
        "path": str(NLP_METRICS_PATH.relative_to(PROJECT_ROOT)),
        "role_in_submission": "Supporting evidence for claim-language scoring"
    },
    {
        "artifact": "Receipt CV metrics",
        "path": str(CV_METRICS_PATH.relative_to(PROJECT_ROOT)),
        "role_in_submission": "Supporting evidence for retained receipt-model research assets"
    },
    {
        "artifact": "Insurance API regression tests",
        "path": "tests/test_api_insurance.py",
        "role_in_submission": "Primary verification of the active end-to-end workflow"
    }
])
evaluation_inventory